# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from lightgbm import LGBMClassifier

from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold, cross_val_score

from feature_engine.encoding import RareLabelEncoder
from feature_engine.selection import DropFeatures

/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def dump_pickle(file_obj, file_path):
    with open(file_path, 'bw') as file:
        pickle.dump(file_obj, file)

# Loading Dataset

In [3]:
X_train = pd.read_parquet('../data/X_train.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

In [4]:
X_train.head()

,driver,compound,race,year,pitstop,lapnumber,stint,tyrelife,position,laptime_s,...,treeraceprogress,treeposition,treelaptime_s,treecumulative_degradation,treeraceprogress_position,treeraceprogress_laptime_s,treeraceprogress_cumulative_degradation,treeposition_laptime_s,treeposition_cumulative_degradation,treelaptime_s_cumulative_degradation
id,,,,,,,,,,,,,,,,,,,,,
0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,...,0.298107,0.207392,0.252153,0.279021,0.298107,0.298107,0.321361,0.252153,0.279021,0.253721
1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,...,0.298107,0.185508,0.252153,0.586119,0.298107,0.298107,0.610176,0.252153,0.544023,0.613159
2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,...,0.298107,0.207392,0.252153,0.207676,0.298107,0.298107,0.423519,0.252153,0.228194,0.231339
3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,...,0.102471,0.185508,0.178575,0.138352,0.102471,0.102471,0.043550,0.178575,0.138352,0.070991
4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,...,0.298107,0.185508,0.178575,0.138352,0.298107,0.298107,0.206543,0.178575,0.138352,0.153276


# Machine Learning

## Base

In [8]:
cv_results = cross_val_score(
    LGBMClassifier(
        random_state=42, 
        verbose=0, 
        class_weight='balanced',
    ),
    X=X_train, 
    y=y_train.PitNextLap, 
    scoring='roc_auc', 
    cv=StratifiedKFold(random_state=42, shuffle=True), 
    n_jobs=-1
)

In [9]:
cv_results.mean()

np.float64(0.9413265345598075)

## Feature Selection

In [10]:
model = LGBMClassifier(random_state=42, verbose=0, class_weight='balanced')
model.fit(X_train, y_train.PitNextLap)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [11]:
perm_result = permutation_importance(
    estimator=model, 
    X=X_train, 
    y=y_train.PitNextLap, 
    n_jobs=-1, 
    scoring='roc_auc'
)


importance_df = pd.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

In [12]:
importance_df.query("importance_mean <= 0").feature.tolist()

['laps_since_pit',
 'position_change_cum',
 'position_norm',
 'lapnumber_high',
 'race_progress_squared',
 'is_pit_lap',
 'stint_progress',
 'lapnumber_low',
 'position_high',
 'stint_low',
 'driver_avg_position',
 'position_medium',
 'laptime_s_high',
 'raceprogress_low',
 'stint_high',
 'position_low',
 'treeraceprogress_laptime_s',
 'treeposition',
 'fuzzy_2_raceprogress_lapnumber',
 'treeraceprogress',
 'treelaptime_s',
 'treeraceprogress_position',
 'treeposition_laptime_s',
 'compound_tyre_life']

In [13]:
features_to_drop = importance_df.query("importance_mean <= 0").feature.tolist()

## Fine Tuning

In [ ]:
def objective(trial, X, y):
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]
        
        model = make_pipeline(
            DropFeatures(features_to_drop),
            LGBMClassifier(
                objective='binary',
                metric='auc',
                boosting_type='gbdt',
                num_leaves=trial.suggest_int('num_leaves', 16, 256),
                max_depth=trial.suggest_int('max_depth', 3, 12),
                learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
                lambda_l1=trial.suggest_float('lambda_l1', 1e-3, 10.0, log=True),
                lambda_l2=trial.suggest_float('lambda_l2', 1e-3, 10.0, log=True),
                feature_fraction=trial.suggest_float('feature_fraction', 0.6, 1.0),
                bagging_fraction=trial.suggest_float('bagging_fraction', 0.6, 1.0),
                bagging_freq=trial.suggest_int('bagging_freq', 1, 7),
                min_child_samples=trial.suggest_int('min_child_samples', 10, 100),
                verbosity=-1,
                n_estimators=2000,
                random_state=42,
                n_jobs=1
            )
        ).fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]
        
        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)


study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30, n_jobs=-1, show_progress_bar=True)

[I 2026-05-10 21:11:26,021] A new study created in memory with name: no-name-61691562-6060-4f36-8da8-fb4ef88ac0f3
  0%|                                                                                          | 0/30 [00:00<?, ?it/s]

In [ ]:
pipe_tuned = make_pipeline(
    DropFeatures(features_to_drop),
    LGBMClassifier(
        objective='binary',
        metric='auc',
        boosting_type='gbdt',
        verbosity=-1,
        n_estimators=2000,
        random_state=42,
        **study.best_params
    )
).fit(X_train, y_train.PitNextLap)

# Deploy

In [13]:
dump_pickle(pipe_tuned, '../models/model_lightgbm.pkl')